In [5]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------
df = pd.read_csv(
    r"C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\Walmart_modified_with_anomalies.csv"
)

print("Initial Shape:", df.shape)

# ------------------------------------------------------------
# 2. DATE PARSING
# ------------------------------------------------------------
if 'Purchase_Date' in df.columns:
    df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'], errors='coerce')

# ------------------------------------------------------------
# 3. REMOVE DUPLICATES
# ------------------------------------------------------------
df.drop_duplicates(inplace=True)
print("After removing duplicates:", df.shape)

# ------------------------------------------------------------
# 4. MISSING VALUE HANDLING
# ------------------------------------------------------------
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns

# Numeric → median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical → mode (no encoding)
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values handled")

# ------------------------------------------------------------
# 5. NEGATIVE VALUE HANDLING (Business Rule)
# ------------------------------------------------------------
for col in num_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"{col}: {neg_count} negative values fixed")
        df[col] = df[col].clip(lower=0)

# ------------------------------------------------------------
# 6. OUTLIER TREATMENT (IQR METHOD)
# ------------------------------------------------------------
outlier_cols = [
    'Market_Price',
    'Purchase_Amount',
    'Competitor_Rating',
    'Competitor_Price'
]

for col in outlier_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = df[col].clip(lower, upper)

print("Outliers handled using IQR")

# ------------------------------------------------------------
# 7. SORT FOR TIME SERIES (Agent 1)
# ------------------------------------------------------------
if 'Purchase_Date' in df.columns:
    df.sort_values('Purchase_Date', inplace=True)

# ------------------------------------------------------------
# 8. FINAL CHECK
# ------------------------------------------------------------
print("\nFinal Shape:", df.shape)
print("\nRemaining Missing Values:\n", df.isna().sum())

# ------------------------------------------------------------
# 9. SAVE PREPROCESSED DATA
# ------------------------------------------------------------
output_path = r"C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\walmart_preprocessed_basic.csv"
df.to_csv(output_path, index=False)

print("\n✅ BASIC PREPROCESSING COMPLETED")
print("Saved at:", output_path)


Initial Shape: (52500, 20)
After removing duplicates: (50187, 20)
Missing values handled
Outliers handled using IQR

Final Shape: (50187, 20)

Remaining Missing Values:
 Product_ID                 0
Product_Name               0
Brand                      0
Category                   0
Market_Price               0
Purchase_Amount            0
Discount_Applied           0
Rating                     0
Customer_ID                0
Age                        0
Gender                     0
City                       0
Purchase_Date           1031
Payment_Method             0
Repeat_Customer            0
Competitor_Name            0
Competitor_Price           0
Market_Share               0
Competitor_Rating          0
Promotion_Competitor       0
dtype: int64

✅ BASIC PREPROCESSING COMPLETED
Saved at: C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\walmart_preprocessed_basic.csv
